In [3]:
%%capture
!pip install keras-cv

In [4]:
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


In [5]:
import os
os.environ["KERAS_BACKEND"] = "tensorflow" # other options: tensorflow or torch
import json
import keras_cv
import keras
from keras import ops
import tensorflow as tf

import cv2
import pandas as pd
import numpy as np
from glob import glob
from tqdm.notebook import tqdm
import joblib

import matplotlib.pyplot as plt

In [6]:
class CFG:
    verbose = 1  # Verbosity
    seed = 47  # Random seed
    neg_sample = 0.01 # Downsample negative calss
    pos_sample = 5.0  # Upsample positive class
    preset = "efficientnetv2_b2_imagenet"  # Name of pretrained classifier
    image_size = [128, 128]  # Input image size
    epochs = 10 # Training epochs
    batch_size = 256  # Batch size
    lr_mode = "cos" # LR scheduler mode from one of "cos", "step", "exp"
    class_names = ['target']
    num_classes = 1

keras.utils.set_random_seed(CFG.seed)

In [7]:
ROOT = str(os.getcwd())
BASE_PATH = f"{ROOT}/drive/MyDrive/WAIT"
DATA_PATH = f"{BASE_PATH}/data/new"
MODEL_PATH = f"{BASE_PATH}/models/enhanced/"
PROC_PATH = f"{MODEL_PATH}/enhanced/preprocessed_dataset/"

In [8]:
# Train + Valid
df = pd.read_parquet(f'{DATA_PATH}/enhanced/enhanced_train_metadata_clean.parquet')

# Testing
testing_df = pd.read_parquet(f'{DATA_PATH}/enhanced/enhanced_test_metadata_clean.parquet')

In [9]:
print("Class Distribution Before Sampling (%):")
display(df.target.value_counts(normalize=True)*100)

# Sampling
positive_df = df.query("target==0").sample(frac=CFG.neg_sample, random_state=CFG.seed)
negative_df = df.query("target==1").sample(frac=CFG.pos_sample, replace=True, random_state=CFG.seed)
df = pd.concat([positive_df, negative_df], axis=0).sample(frac=1.0)

print("\nCalss Distribution After Sampling (%):")
display(df.target.value_counts(normalize=True)*100)

Class Distribution Before Sampling (%):


,proportion
target,
0,99.899858
1,0.100142



Calss Distribution After Sampling (%):


,proportion
target,
0,66.611631
1,33.388369


In [10]:
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight('balanced', classes=np.unique(df['target']), y=df['target'])
class_weights = dict(enumerate(class_weights))
print("Class Weights:", class_weights)

Class Weights: {0: np.float64(0.7506196640044065), 1: np.float64(1.4975274725274725)}


In [11]:
import h5py

training_validation_hdf5 = h5py.File(f"{DATA_PATH}/train-image.hdf5", 'r')
testing_hdf5 = h5py.File(f"{DATA_PATH}/test-image.hdf5", 'r')

In [12]:
from sklearn.model_selection import StratifiedGroupKFold

df = df.reset_index(drop=True) # ensure continuous index
df["fold"] = -1
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=CFG.seed)
for i, (training_idx, validation_idx) in enumerate(sgkf.split(df, y=df.target, groups=df.patient_id)):
    df.loc[validation_idx, "fold"] = int(i)

# Use first fold for training and validation
training_df = df.query("fold!=0")
validation_df = df.query("fold==0")
print(f"# Num Train: {len(training_df)} | Num Valid: {len(validation_df)}")

# Num Train: 4383 | Num Valid: 1068


In [13]:
# class distribution in training and validation
print("training:", training_df.target.value_counts(),
      "\nvalidation:", validation_df.target.value_counts(), sep='\n')

training:
target
0    2929
1    1454
Name: count, dtype: int64

validation:
target
0    702
1    366
Name: count, dtype: int64


# 📊 | Tabular Features

### Enhanced


In [14]:
# Categorical features which will be one hot encoded
CATEGORICAL_COLUMNS = ["sex", "anatom_site_general", "tbp_tile_type",
                       "tbp_lv_location", "attribution"]

NEW_CATEGORICAL_COLUMNS = ["combined_anatomical_site"]

# Numerical features which will be normalized
NUMERIC_COLUMNS = ["age_approx", "tbp_lv_nevi_confidence", "clin_size_long_diam_mm",
           "tbp_lv_areaMM2", "tbp_lv_area_perim_ratio", "tbp_lv_color_std_mean",
           "tbp_lv_deltaLBnorm", "tbp_lv_minorAxisMM"]

NEW_NUMERIC_COLUMNS = ["lesion_size_ratio", "lesion_shape_index", "hue_contrast",
        "luminance_contrast", "lesion_color_difference", "border_complexity",
         "3d_position_distance", "perimeter_to_area_ratio",
        "lesion_visibility_score", "symmetry_border_consistency", "color_consistency",

        "size_age_interaction", "hue_color_std_interaction", "lesion_severity_index",
        "shape_complexity_index", "color_contrast_index", "log_lesion_area",
        "normalized_lesion_size", "mean_hue_difference", "std_dev_contrast",
        "color_shape_composite_index", "3d_lesion_orientation", "overall_color_difference",
        "symmetry_perimeter_interaction", "comprehensive_lesion_index"]

# Tabular feature columns
FEAT_COLS = (CATEGORICAL_COLUMNS + NEW_CATEGORICAL_COLUMNS) + (NUMERIC_COLUMNS + NEW_NUMERIC_COLUMNS)

In [15]:
def build_augmenter():
    # Define augmentations
    aug_layers = [
        keras_cv.layers.RandomCutout(height_factor=(0.02, 0.06), width_factor=(0.02, 0.06)),
        keras_cv.layers.RandomFlip(mode="horizontal"),
    ]

    # Apply augmentations to random samples
    aug_layers = [keras_cv.layers.RandomApply(x, rate=0.5) for x in aug_layers]

    # Build augmentation layer
    augmenter = keras_cv.layers.Augmenter(aug_layers)

    # Apply augmentations
    def augment(inp, label):
        images = inp["images"]
        aug_data = {"images": images}
        aug_data = augmenter(aug_data)
        inp["images"] = aug_data["images"]
        return inp, label
    return augment


def build_decoder(with_labels=True, target_size=CFG.image_size):
    def decode_image(inp):
        # Read jpeg image
        file_bytes = inp["images"]
        image = tf.io.decode_jpeg(file_bytes)

        # Resize
        image = tf.image.resize(image, size=target_size, method="area")

        # Rescale image
        image = tf.cast(image, tf.float32)
        image /= 255.0

        # Reshape
        image = tf.reshape(image, [*target_size, 3])

        inp["images"] = image
        return inp

    def decode_label(label, num_classes):
        label = tf.cast(label, tf.float32)
        label = tf.reshape(label, [num_classes])
        return label

    def decode_with_labels(inp, label=None):
        inp = decode_image(inp)
        label = decode_label(label, CFG.num_classes)
        return (inp, label)

    return decode_with_labels if with_labels else decode_image


def build_dataset(
    isic_ids,
    hdf5,
    features,
    labels=None,
    batch_size=32,
    decode_fn=None,
    augment_fn=None,
    augment=False,
    shuffle=1024,
    cache=True,
    drop_remainder=False,
):
    if decode_fn is None:
        decode_fn = build_decoder(labels is not None)

    if augment_fn is None:
        augment_fn = build_augmenter()

    AUTO = tf.data.experimental.AUTOTUNE

    images = [None]*len(isic_ids)
    for i, isic_id in enumerate(tqdm(isic_ids, desc="Loading Images ")):
        images[i] = hdf5[isic_id][()]

    inp = {"images": images, "features": features}
    slices = (inp, labels) if labels is not None else inp

    ds = tf.data.Dataset.from_tensor_slices(slices)
    ds = ds.cache() if cache else ds
    ds = ds.map(decode_fn, num_parallel_calls=AUTO)
    if shuffle:
        ds = ds.shuffle(shuffle, seed=CFG.seed)
        opt = tf.data.Options()
        opt.deterministic = False
        ds = ds.with_options(opt)
    ds = ds.batch(batch_size, drop_remainder=drop_remainder)
    ds = ds.map(augment_fn, num_parallel_calls=AUTO) if augment else ds
    ds = ds.prefetch(AUTO)
    return ds

In [16]:
# Build Training & Validation Dataset
print("# Training:")
training_features = dict(training_df[FEAT_COLS])
training_ids = training_df.isic_id.values
training_labels = training_df.target.values
training_ds = build_dataset(training_ids, training_validation_hdf5, training_features,
                         training_labels, batch_size=CFG.batch_size,
                         shuffle=True, augment=False)

print("# Validation:")
validation_features = dict(validation_df[FEAT_COLS])
validation_ids = validation_df.isic_id.values
validation_labels = validation_df.target.values
validation_ds = build_dataset(validation_ids, training_validation_hdf5, validation_features,
                         validation_labels, batch_size=CFG.batch_size,
                         shuffle=False, augment=False)

# Testing
print("# Testing:")
testing_features = dict(testing_df[FEAT_COLS])
testing_ids = testing_df.isic_id.values
testing_ds = build_dataset(testing_ids, testing_hdf5,
                        testing_features, batch_size=CFG.batch_size,
                         shuffle=False, augment=False, cache=False)

# Training:


Loading Images :   0%|          | 0/4383 [00:00<?, ?it/s]

# Validation:


Loading Images :   0%|          | 0/1068 [00:00<?, ?it/s]

# Testing:


Loading Images :   0%|          | 0/37576 [00:00<?, ?it/s]

In [17]:
feature_space = keras.utils.FeatureSpace(
    features={
        # Categorical features encoded as integers
        "sex": "string_categorical",
        "anatom_site_general": "string_categorical",
        "tbp_tile_type": "string_categorical",
        "tbp_lv_location": "string_categorical",
        "combined_anatomical_site": "string_categorical",
        "attribution": "string_categorical",
        # Numerical features to discretize
        "age_approx": "float_discretized",
        # Numerical features to normalize
        "tbp_lv_nevi_confidence": "float_normalized",
        "clin_size_long_diam_mm": "float_normalized",
        "tbp_lv_areaMM2": "float_normalized",
        "tbp_lv_area_perim_ratio": "float_normalized",
        "tbp_lv_color_std_mean": "float_normalized",
        "tbp_lv_deltaLBnorm": "float_normalized",
        "tbp_lv_minorAxisMM": "float_normalized",
        "lesion_size_ratio" : "float_normalized",
        "lesion_shape_index" : "float_normalized",
        "hue_contrast" : "float_normalized",
        "luminance_contrast": "float_normalized",
        "lesion_color_difference": "float_normalized",
        "border_complexity": "float_normalized",
        "3d_position_distance": "float_normalized",
        "perimeter_to_area_ratio": "float_normalized",
        "lesion_visibility_score": "float_normalized",
        "symmetry_border_consistency": "float_normalized",
        "color_consistency": "float_normalized",
        "size_age_interaction": "float_normalized",
        "hue_color_std_interaction": "float_normalized",
        "lesion_severity_index": "float_normalized",
        "shape_complexity_index": "float_normalized",
        "color_contrast_index": "float_normalized",
        "log_lesion_area": "float_normalized",
        "mean_hue_difference": "float_normalized",
        "std_dev_contrast": "float_normalized",
        "color_shape_composite_index": "float_normalized",
        "3d_lesion_orientation": "float_normalized",
        "overall_color_difference": "float_normalized",
        "symmetry_perimeter_interaction": "float_normalized",
        "comprehensive_lesion_index": "float_normalized",
        # numerical features already normalized
        "normalized_lesion_size": "float",
        },
    output_mode="concat",
)

In [18]:
training_ds_with_no_labels = training_ds.map(lambda x, _: x["features"])
feature_space.adapt(training_ds_with_no_labels)
print(feature_space.get_encoded_features().shape)

(None, 128)


In [19]:
# fs_dir = os.path.join(MODEL_PATH, "model-metadata")
# os.makedirs(fs_dir, exist_ok=True)
# filepath = os.path.join(fs_dir, "feature_space_test.keras")

# feature_space.save(filepath)
# print(f"FeatureSpace saved at: '{filepath}'")

In [20]:
# Apply Feature Processing

training_ds = training_ds.map(
    lambda x, y: ({"images": x["images"],
                   "features": feature_space(x["features"])}, y), num_parallel_calls=tf.data.AUTOTUNE)

validation_ds = validation_ds.map(
    lambda x, y: ({"images": x["images"],
                   "features": feature_space(x["features"])}, y), num_parallel_calls=tf.data.AUTOTUNE)

testing_ds = testing_ds.map(
    lambda x: {"images": x["images"],
               "features": feature_space(x["features"])}, num_parallel_calls=tf.data.AUTOTUNE)

In [21]:
# loss & metric
auc = keras.metrics.AUC()
loss = keras.losses.BinaryCrossentropy(label_smoothing=0.02)

In [22]:
# Define input layers
image_input = keras.Input(shape=(*CFG.image_size, 3), name="images")
feat_input = keras.Input(shape=(feature_space.get_encoded_features().shape[1],), name="features")
inp = {"images":image_input, "features":feat_input}

# Branch for image input
backbone = keras_cv.models.EfficientNetV2Backbone.from_preset(CFG.preset)
x1 = backbone(image_input)
x1 = keras.layers.GlobalAveragePooling2D()(x1)
x1 = keras.layers.Dropout(0.2)(x1)

# Branch for tabular/feature input
x2 = keras.layers.Dense(96, activation="selu")(feat_input)
x2 = keras.layers.Dense(128, activation="selu")(x2)
x2 = keras.layers.Dropout(0.1)(x2)

# Concatenate both branches
concat = keras.layers.Concatenate()([x1, x2])

# Output layer
out = keras.layers.Dense(1, activation="sigmoid", dtype="float32")(concat)

# Build model
model = keras.models.Model(inp, out)

# Compile the model
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss=loss,
    metrics=[auc],
)

# Model Summary
model.summary()

100%|██████████| 1.79k/1.79k [00:00<00:00, 3.77MB/s]


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ images (InputLayer) │ (None, 128, 128,  │          0 │ -                 │
│                     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ features            │ (None, 128)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ efficient_net_v2b2… │ (None, 4, 4,      │  8,769,374 │ images[0][0]      │
│ (EfficientNetV2Bac… │ 1408)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 96)        │     12,384 │ features[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 1408)      │          0 │ efficient_net_v2… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 128)       │     12,416 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 1408)      │          0 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 128)       │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 1536)      │          0 │ dropout[0][0],    │
│ (Concatenate)       │                   │            │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 1)         │      1,537 │ concatenate[0][0] │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 8,795,711 (33.55 MB)

 Trainable params: 8,713,423 (33.24 MB)

 Non-trainable params: 82,288 (321.44 KB)

In [23]:
# keras.utils.plot_model(model, show_shapes=True, show_layer_names=True, dpi=60)

In [24]:
import math

def get_lr_callback(batch_size=8, mode='cos', epochs=10, plot=False):
    lr_start, lr_max, lr_min = 2.5e-5, 5e-6 * batch_size, 0.8e-5
    lr_ramp_ep, lr_sus_ep, lr_decay = 3, 0, 0.75

    def lrfn(epoch):  # Learning rate update function
        if epoch < lr_ramp_ep: lr = (lr_max - lr_start) / lr_ramp_ep * epoch + lr_start
        elif epoch < lr_ramp_ep + lr_sus_ep: lr = lr_max
        elif mode == 'exp': lr = (lr_max - lr_min) * lr_decay**(epoch - lr_ramp_ep - lr_sus_ep) + lr_min
        elif mode == 'step': lr = lr_max * lr_decay**((epoch - lr_ramp_ep - lr_sus_ep) // 2)
        elif mode == 'cos':
            decay_total_epochs, decay_epoch_index = epochs - lr_ramp_ep - lr_sus_ep + 3, epoch - lr_ramp_ep - lr_sus_ep
            phase = math.pi * decay_epoch_index / decay_total_epochs
            lr = (lr_max - lr_min) * 0.5 * (1 + math.cos(phase)) + lr_min
        return lr

    if plot:  # Plot lr curve if plot is True
        plt.figure(figsize=(10, 5))
        plt.plot(np.arange(epochs), [lrfn(epoch) for epoch in np.arange(epochs)], marker='o')
        plt.xlabel('epoch'); plt.ylabel('lr')
        plt.title('LR Scheduler')
        plt.show()

    return keras.callbacks.LearningRateScheduler(lrfn, verbose=False)  # Create lr callback

lr_cb = get_lr_callback(CFG.batch_size, mode="exp", plot=False)

In [25]:
%%capture
!pip install --no-cache-dir tf-explain

In [26]:
raise StopExecution

NameError: name 'StopExecution' is not defined

In [27]:
import numpy as np
import tensorflow as tf


def apply_grey_patch(image, top_left_x, top_left_y, patch_size):
    """
    Replace a part of the image with a grey patch.

    Args:
        image (numpy.ndarray): Input image
        top_left_x (int): Top Left X position of the applied box
        top_left_y (int): Top Left Y position of the applied box
        patch_size (int): Size of patch to apply

    Returns:
        numpy.ndarray: Patched image
    """
    patched_image = np.array(image, copy=True)
    patched_image[
        top_left_y : top_left_y + patch_size, top_left_x : top_left_x + patch_size, :
    ] = 127.5

    return patched_image


@tf.function
def transform_to_normalized_grayscale(tensor):
    """ Transform tensor over RGB axis to grayscale and apply min-max normalization per image. """
    grayscale_tensor = tf.reduce_sum(tensor, axis=-1)

    # min-max normalization per image
    v_min = tf.reduce_min(grayscale_tensor, axis=(1, 2), keepdims=True)
    v_max = tf.reduce_max(grayscale_tensor, axis=(1, 2), keepdims=True)
    normalized_tensor = (grayscale_tensor - v_min) / (v_max - v_min + 1e-8)

    return normalized_tensor


def grid_display(array, num_rows=None, num_columns=None):
    """
    Display a list of images as a grid.

    Args:
        array (numpy.ndarray): 4D Tensor (batch_size, height, width, channels)

    Returns:
        numpy.ndarray: 3D Tensor as concatenation of input images on a grid
    """
    if num_rows is not None and num_columns is not None:
        total_grid_size = num_rows * num_columns
        if total_grid_size < len(array):
            warnings.warn(
                Warning(
                    "Given values for num_rows and num_columns doesn't allow to display "
                    "all images. Values have been overrided to respect at least num_columns"
                )
            )
            num_rows = math.ceil(len(array) / num_columns)
    elif num_rows is not None:
        num_columns = math.ceil(len(array) / num_rows)
    elif num_columns is not None:
        num_rows = math.ceil(len(array) / num_columns)
    else:
        num_rows = math.ceil(math.sqrt(len(array)))
        num_columns = math.ceil(math.sqrt(len(array)))

    number_of_missing_elements = num_columns * num_rows - len(array)
    # We fill the array with np.zeros elements to obtain a perfect square
    array = np.append(
        array,
        np.zeros((number_of_missing_elements, *array[0].shape)).astype(array.dtype),
        axis=0,
    )

    grid = np.concatenate(
        [
            np.concatenate(
                array[index * num_columns : (index + 1) * num_columns], axis=1
            )
            for index in range(num_rows)
        ],
        axis=0,
    )

    return grid

In [28]:
import os
from pathlib import Path
from typing import Optional, Tuple, List

import cv2
import numpy as np
import tensorflow as tf
from keras import Model
from keras.callbacks import Callback


class SmoothGradEngine:
    """
    SmoothGrad for multimodal models (images + tabular features).
    
    Uses pure input-space gradients (tf-explain approach):
    - Generate noisy versions of input
    - Compute gradients w.r.t. input  
    - Average gradients across noisy samples
    - Convert to grayscale saliency map
    """
    
    def explain(
        self,
        validation_data: Tuple[dict, np.ndarray],
        model: tf.keras.Model,
        class_index: int,
        num_samples: int = 50,
        noise: float = 0.15,
    ) -> np.ndarray:
        """
        Compute SmoothGrad saliency maps for multimodal input.
        
        Algorithm:
        1. For each image in batch:
            - Generate num_samples noisy versions
            - Compute gradients w.r.t. input for each noisy version
            - Average gradients across samples
            - Normalize to [0, 1]
        
        Args:
            validation_data: ({'images': (B,H,W,3), 'features': (B,F)}, labels)
            model: Keras model with inputs ['images', 'features']
            class_index: target class index (0 or 1 for binary)
            num_samples: number of noisy samples to average (default 50)
            noise: Gaussian noise std (default 0.15)
        
        Returns:
            saliency_maps: (B, H, W, 1) float32 in [0, 1]
        """
        x, labels = validation_data
        images = x['images']  # (B, H, W, 3)
        features = x['features']  # (B, F)
        
        batch_size = images.shape[0]
        
        saliency_maps_list = []
        
        for img_idx in range(batch_size):
            # Extract single image and features
            img_single = images[img_idx:img_idx+1]  # (1, H, W, 3)
            feat_single = features[img_idx:img_idx+1]  # (1, F)
            
            # Generate num_samples noisy versions
            noisy_images = self.generate_noisy_images(
                img_single.numpy(), 
                num_samples, 
                noise
            )  # (num_samples, H, W, 3)
            
            # Compute gradients for each noisy sample
            gradients_list = []
            
            for sample_idx in range(num_samples):
                img_noisy = tf.cast(noisy_images[sample_idx:sample_idx+1], tf.float32)
                
                with tf.GradientTape() as tape:
                    tape.watch(img_noisy)
                    
                    # Forward pass through model
                    preds = model({'images': img_noisy, 'features': feat_single}, training=False)
                    
                    # Compute loss w.r.t. target class
                    if preds.shape[-1] == 1:
                        # Binary classification
                        loss = -tf.math.log(tf.clip_by_value(preds[:, 0], 1e-7, 1.0))
                    else:
                        # Multiclass
                        loss = -tf.math.log(
                            tf.clip_by_value(preds[:, class_index], 1e-7, 1.0)
                        )
                
                # Compute gradients w.r.t. input images
                grads = tape.gradient(loss, img_noisy)
                
                # Validation: check for None/NaN
                if grads is None:
                    raise ValueError(
                        "Gradient is None - tape not properly set up. "
                        "Ensure model inputs are properly named."
                    )
                
                if tf.math.reduce_any(tf.math.is_nan(grads)):
                    raise ValueError("NaN detected in gradients")
                
                gradients_list.append(grads.numpy())
            
            # Stack and average gradients across noisy samples
            grads_stacked = np.stack(gradients_list, axis=0)  # (num_samples, 1, H, W, 3)
            grads_avg = grads_stacked.mean(axis=0)[0]  # (H, W, 3)
            
            # Convert to grayscale saliency
            saliency = np.abs(grads_avg).mean(axis=-1)  # (H, W)
            
            # Normalize to [0, 1]
            saliency_min = saliency.min()
            saliency_max = saliency.max()
            if saliency_max > saliency_min:
                saliency_norm = (saliency - saliency_min) / (saliency_max - saliency_min)
            else:
                saliency_norm = np.zeros_like(saliency)
            
            saliency_maps_list.append(saliency_norm[..., np.newaxis])
        
        # Stack into batch
        saliency_batch = np.stack(saliency_maps_list, axis=0)  # (B, H, W, 1)
        
        return saliency_batch.astype(np.float32)
    
    @staticmethod
    def generate_noisy_images(
        images: np.ndarray, 
        num_samples: int, 
        noise: float
    ) -> np.ndarray:
        """Generate num_samples noisy copies by adding Gaussian noise."""
        repeated = np.repeat(images, num_samples, axis=0)
        noise_array = np.random.normal(0, noise, repeated.shape).astype(np.float32)
        noisy = repeated + noise_array
        return np.clip(noisy, 0, 1).astype(np.float32)


class SmoothGradCallback(Callback):
    """Keras callback to save SmoothGrad visualizations each epoch."""
    
    def __init__(
        self,
        engine: SmoothGradEngine,
        probe_batch: Tuple[dict, np.ndarray],
        class_index: int = 0,
        save_dir: Optional[str] = None,
        num_samples: int = 50,
        noise: float = 0.15,
    ):
        super().__init__()
        self.engine = engine
        self.probe_batch = probe_batch
        self.class_index = class_index
        self.save_dir = Path(save_dir) if save_dir else None
        self.num_samples = num_samples
        self.noise = noise
        
        if self.save_dir:
            self.save_dir.mkdir(parents=True, exist_ok=True)
    
    def on_epoch_end(self, epoch, logs=None):
        """Called at end of each epoch."""
        # Compute SmoothGrad
        saliency_maps = self.engine.explain(
            validation_data=self.probe_batch,
            model=self.model,
            class_index=self.class_index,
            num_samples=self.num_samples,
            noise=self.noise,
        )  # (B, H, W, 1)
        
        # Visualize as grid
        grid = self._make_grid(saliency_maps)
        
        # Save to disk
        if self.save_dir:
            output_path = self.save_dir / f"smoothgrad_epoch_{epoch:03d}.png"
            grid_uint8 = np.uint8(np.clip(grid, 0, 1) * 255)
            grid_bgr = cv2.cvtColor(grid_uint8, cv2.COLOR_RGB2BGR)
            cv2.imwrite(str(output_path), grid_bgr)
            print(f"Epoch {epoch}: SmoothGrad saved to {output_path}")
    
    @staticmethod
    def _make_grid(saliency_maps: np.ndarray) -> np.ndarray:
        """Arrange saliency maps in a grid for visualization."""
        batch_size, H, W, _ = saliency_maps.shape
        num_cols = min(4, batch_size)
        num_rows = (batch_size + num_cols - 1) // num_cols
        
        maps_uint8 = np.uint8(np.clip(saliency_maps[..., 0], 0, 1) * 255)
        maps_rgb = np.stack([maps_uint8] * 3, axis=-1)
        
        rows = []
        for r in range(num_rows):
            start_idx = r * num_cols
            end_idx = min(start_idx + num_cols, batch_size)
            row_images = maps_rgb[start_idx:end_idx]
            
            if len(row_images) < num_cols:
                padding = np.zeros((num_cols - len(row_images), H, W, 3), dtype=np.uint8)
                row_images = np.vstack([row_images, padding])
            
            row = np.hstack(row_images)
            rows.append(row)
        
        grid = np.vstack(rows)
        return grid.astype(np.float32) / 255.0



In [ ]:
XAI_CB_PATH = os.path.join(MODEL_PATH, 'xai', 'test', 'no_aug' , 'callback', 'smoothgrad')
base_dir=os.path.join(XAI_CB_PATH, "images")
os.makedirs(base_dir, exist_ok=True)

x, labels = next(iter(validation_ds.skip(0).take(1)))

sg_cb = MultiModalSmoothGradCallback(
    engine=MultiModalSmoothGradEngine(use_guided_grads=True, mode="image_only"),
    probe_batch=(x, labels),
    class_index=0,
    save_dir=base_dir)

In [ ]:
model_fpath = os.path.join(MODEL_PATH, 'saved_models', 'best_enhanced_model_test_noaug.keras')

ckpt_cb = keras.callbacks.ModelCheckpoint(
    model_fpath,
    monitor="val_auc",
    save_best_only=True,
    save_weights_only=False,
    mode="max",
)

In [ ]:
cbs = [lr_cb, ckpt_cb, sg_cb]

history = model.fit(
    training_ds,
    epochs=4,
    callbacks=cbs,
    validation_data=validation_ds,
    verbose=CFG.verbose,
    class_weight=class_weights,
)

In [ ]:
raise stopExecution

In [ ]:
# %reload_ext tensorboard
# %tensorboard --logdir "{os.path.join(XAI_PATH, 'tb')}"

In [ ]:
# model.save(model_fpath, include_optimizer=True)

test flip

In [ ]:
# 1) Instantiate the engine
engine = MultiModalSmoothGradEngine(
    use_guided_grads=True,        # guided-ReLU style gradients
    mode='context',               # 'context' or 'image_only'
    baseline=None,                # for image_only; defaults to zeros per feature width
)

In [ ]:
test_path = '/content/drive/MyDrive/WAIT/models/enhanced/xai/smoothgrad/nuovi_test/debug/callback'

(x0, y0) = next(iter(validation_ds.take(1)))

grid_o, triptychs_o, _ = engine.explain((x0, y0), model=model,
                                        num_samples=5,class_index=0)
Ht, Wt, Ct = triptychs_o[0].shape

x1 = {"images": tf.image.flip_left_right(x0["images"]),
    "features": x0["features"]}

grid_f, _, _ = engine.explain((x1, y0), model=model, class_index=0,
                              num_samples=5)

engine.save_grid(grid_o, test_path, "orig.png")
engine.save_grid(grid_f, test_path, "flipped.png")
print("Triptych (tile) size:", (Ht, Wt, Ct))

In [ ]:
# DEBUG: Trace through the pipeline for a single image to diagnose the issue
import matplotlib.pyplot as plt

(x0_debug, y0_debug) = next(iter(validation_ds.take(1)))

# Take just the first 2 images from the batch
x_debug = {k: v[:2] for k, v in x0_debug.items()}
y_debug = y0_debug[:2]

# Original
img_orig = x_debug['images'][0:1]
feat_orig = x_debug['features'][0:1]

# Flipped
img_flip = tf.image.flip_left_right(img_orig)
feat_flip = feat_orig

# Generate noisy copies manually to inspect
num_samples_debug = 5
noise_debug = 0.1

noisy_orig = engine.generate_noisy_images(img_orig.numpy(), num_samples_debug, noise_debug)
noisy_flip = engine.generate_noisy_images(img_flip.numpy(), num_samples_debug, noise_debug)

print('SANITY CHECKS:')
print(f'img_orig shape: {img_orig.shape}, min/max: {img_orig.min():.6f} / {img_orig.max():.6f}')
print(f'img_flip shape: {img_flip.shape}, min/max: {img_flip.min():.6f} / {img_flip.max():.6f}')
print(f'Are img_orig and img_flip identical? {np.allclose(img_orig.numpy(), img_flip.numpy(), atol=1e-6)}')
print(f'img_orig[0,0,0,0] = {img_orig.numpy()[0,0,0,0]:.6f}, img_flip[0,0,0,0] = {img_flip.numpy()[0,0,0,0]:.6f}')
print(f'img_orig[0,0,-1,0] = {img_orig.numpy()[0,0,-1,0]:.6f}, img_flip[0,0,-1,0] = {img_flip.numpy()[0,0,-1,0]:.6f}')
print(f'noisy_orig[0] min/max: {noisy_orig[0].min():.6f} / {noisy_orig[0].max():.6f}')
print(f'noisy_flip[0] min/max: {noisy_flip[0].min():.6f} / {noisy_flip[0].max():.6f}')
print(f'Are noisy_orig and noisy_flip identical? {np.allclose(noisy_orig, noisy_flip, atol=1e-6)}')
print(f'Are noisy_flip and flipped(noisy_orig) the same? {np.allclose(noisy_flip, noisy_orig[:, :, ::-1, :], atol=1e-6)}')
print()

# Compute gradients for each noisy sample separately (no averaging yet)
print('Computing raw gradients for each sample...')
grads_orig_samples = []
grads_flip_samples = []

for i in range(num_samples_debug):
    # Original
    img_i = tf.cast(noisy_orig[i:i+1], tf.float32)
    feat_i = tf.repeat(feat_orig, 1, axis=0)
    with tf.GradientTape() as tape:
        tape.watch(img_i)
        preds = model({'images': img_i, 'features': feat_i}, training=False)
        p = tf.clip_by_value(preds[:, 0], 1e-6, 1-1e-6)
        target = tf.math.log(p) - tf.math.log(1.0 - p)
    grad = tape.gradient(target, img_i)
    grads_orig_samples.append(grad.numpy())

    # Flipped
    img_flip_i = tf.cast(noisy_flip[i:i+1], tf.float32)
    with tf.GradientTape() as tape:
        tape.watch(img_flip_i)
        preds_flip = model({'images': img_flip_i, 'features': feat_flip}, training=False)
        p_flip = tf.clip_by_value(preds_flip[:, 0], 1e-6, 1-1e-6)
        target_flip = tf.math.log(p_flip) - tf.math.log(1.0 - p_flip)
    grad_flip = tape.gradient(target_flip, img_flip_i)
    grads_flip_samples.append(grad_flip.numpy())

print(f'Collected {len(grads_orig_samples)} gradient samples for original')
print(f'Collected {len(grads_flip_samples)} gradient samples for flipped')

# Stack and analyze
grads_orig_stack = np.stack(grads_orig_samples, axis=0)  # (5, 1, 128, 128, 3)
grads_flip_stack = np.stack(grads_flip_samples, axis=0)  # (5, 1, 128, 128, 3)

print('\n=== RAW GRADIENTS (before any processing) ===')
print(f'grads_orig shape: {grads_orig_stack.shape}')
print(f'grads_orig min/max: {grads_orig_stack.min():.8f} / {grads_orig_stack.max():.8f}')
print(f'grads_flip min/max: {grads_flip_stack.min():.8f} / {grads_flip_stack.max():.8f}')
print(f'Are raw grads identical? {np.allclose(grads_orig_stack, grads_flip_stack, atol=1e-6)}')

# Check if flipped gradient matches horizontally flipped original gradient
grads_orig_flipped = grads_orig_stack[:, :, :, ::-1, :]  # flip axis 3 (W)
print(f'Do grads_flip match flipped(grads_orig)? {np.allclose(grads_flip_stack, grads_orig_flipped, atol=1e-4)}')

# Now apply L-inf normalization
print('\n=== AFTER L-inf NORMALIZATION ===')
g_max_orig = np.abs(grads_orig_stack).max(axis=(2, 3, 4), keepdims=True) + 1e-8
grads_orig_norm = grads_orig_stack / g_max_orig
g_max_flip = np.abs(grads_flip_stack).max(axis=(2, 3, 4), keepdims=True) + 1e-8
grads_flip_norm = grads_flip_stack / g_max_flip

print(f'After L-inf norm, grads_orig min/max: {grads_orig_norm.min():.6f} / {grads_orig_norm.max():.6f}')
print(f'After L-inf norm, grads_flip min/max: {grads_flip_norm.min():.6f} / {grads_flip_norm.max():.6f}')
print(f'Are normalized grads identical? {np.allclose(grads_orig_norm, grads_flip_norm, atol=1e-6)}')

# Average
print('\n=== AFTER AVERAGING OVER SAMPLES ===')
grad_orig_avg = grads_orig_norm.mean(axis=0)  # (1, 128, 128, 3)
grad_flip_avg = grads_flip_norm.mean(axis=0)  # (1, 128, 128, 3)
print(f'grad_orig_avg shape: {grad_orig_avg.shape}')
print(f'grad_orig_avg min/max: {grad_orig_avg.min():.6f} / {grad_orig_avg.max():.6f}')
print(f'grad_flip_avg min/max: {grad_flip_avg.min():.6f} / {grad_flip_avg.max():.6f}')
print(f'Are averaged grads identical? {np.allclose(grad_orig_avg, grad_flip_avg, atol=1e-6)}')

# Check flip consistency
grad_orig_avg_flipped = grad_orig_avg[:, :, ::-1, :]
print(f'Do averaged_flip match flipped(averaged_orig)? {np.allclose(grad_flip_avg, grad_orig_avg_flipped, atol=1e-5)}')

# Apply abs and grayscale transform
print('\n=== AFTER ABS AND GRAYSCALE TRANSFORM ===')
grad_orig_gray = transform_to_normalized_grayscale(tf.abs(tf.constant(grad_orig_avg, dtype=tf.float32))).numpy()
grad_flip_gray = transform_to_normalized_grayscale(tf.abs(tf.constant(grad_flip_avg, dtype=tf.float32))).numpy()
print(f'grad_orig_gray shape: {grad_orig_gray.shape}')
print(f'grad_orig_gray min/max: {grad_orig_gray.min():.6f} / {grad_orig_gray.max():.6f}')
print(f'grad_flip_gray min/max: {grad_flip_gray.min():.6f} / {grad_flip_gray.max():.6f}')
print(f'Are grayscale maps identical? {np.allclose(grad_orig_gray, grad_flip_gray, atol=1e-6)}')

grad_orig_gray_flipped = grad_orig_gray[:, :, ::-1]
print(f'Do gray_flip match flipped(gray_orig)? {np.allclose(grad_flip_gray, grad_orig_gray_flipped, atol=1e-5)}')

print('\n=== SPATIAL STATISTICS ===')
print(f'grad_orig_gray corner (0,0): {grad_orig_gray[0, 0, 0]:.6f}')
print(f'grad_orig_gray corner (0,-1): {grad_orig_gray[0, 0, -1]:.6f}')
print(f'grad_flip_gray corner (0,0): {grad_flip_gray[0, 0, 0]:.6f}')
print(f'grad_flip_gray corner (0,-1): {grad_flip_gray[0, 0, -1]:.6f}')
print(f'Corner (0,0) values: orig={grad_orig_gray.flatten().max():.6f}, flip={grad_flip_gray.flatten().max():.6f}')
print(f'Max value location in orig: {np.unravel_index(grad_orig_gray.argmax(), grad_orig_gray.shape)}')
print(f'Max value location in flip: {np.unravel_index(grad_flip_gray.argmax(), grad_flip_gray.shape)}')

In [ ]:
print('SANITY CHECKS:')
io = img_orig.numpy(); ifl = img_flip.numpy()
print(f'img_orig shape: {io.shape}, min/max: {io.min():.6f} / {io.max():.6f}')
print(f'img_flip shape: {ifl.shape}, min/max: {ifl.min():.6f} / {ifl.max():.6f}')
print(f'Are img_orig and img_flip identical? {np.allclose(io, ifl, atol=1e-6)}')
print(f'img_orig[0,0,0,0]  = {io[0,0,0,0]:.6f},  img_flip[0,0,0,0]  = {ifl[0,0,0,0]:.6f}')
print(f'img_orig[0,0,-1,0] = {io[0,0,-1,0]:.6f}, img_flip[0,0,-1,0] = {ifl[0,0,-1,0]:.6f}')
print(f'noisy_orig[0] min/max: {noisy_orig[0].min():.6f} / {noisy_orig[0].max():.6f}')
print(f'noisy_flip[0] min/max: {noisy_flip[0].min():.6f} / {noisy_flip[0].max():.6f}')
print(f'Are noisy_orig and noisy_flip identical? {np.allclose(noisy_orig, noisy_flip, atol=1e-6)}')
print(f'Are noisy_flip and flipped(noisy_orig) the same? {np.allclose(noisy_flip, noisy_orig[:, :, ::-1, :], atol=1e-6)}')

# Also check what the model actually outputs for orig vs flipped
p_orig = model({'images': tf.cast(io, tf.float32), 'features': feat_orig}, training=False)
p_flip = model({'images': tf.cast(ifl, tf.float32), 'features': feat_flip}, training=False)
print(f'\nModel pred on orig:    {p_orig.numpy().ravel()}')
print(f'Model pred on flipped: {p_flip.numpy().ravel()}')
print(f'Predictions identical? {np.allclose(p_orig.numpy(), p_flip.numpy(), atol=1e-5)}')

debug

In [ ]:
test_path = '/content/drive/MyDrive/WAIT/models/enhanced/xai/grad_cam/nuovi_test/debug/callback'

n_samp = 15
if n_samp is None:
    n_samp = len(validation_ds)

(x0, y0) = next(iter(validation_ds.take(1)))
x0 = {k: v[:n_samp] for k, v in x0.items()}
y0 = y0[:n_samp]

grid_o, triptychs_o, _ = engine.explain((x0, y0), model=model, class_index=0)

x1 = {"images": tf.image.flip_left_right(x0["images"]), "features": x0["features"]}
grid_f, _, _ = engine.explain((x1, y0), model=model, class_index=0)

engine.save_grid(grid_o, test_path, "orig.png")

print("Triptych (tile) size:", triptychs_o[0].shape)